In [1]:
import glob, os, sys, math, json
import numpy as np
import pandas as pd
import torch
import transformers

DOSSIER = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
sys.path.append(DOSSIER)
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions, chercher_seuils

train = pd.read_csv(f"{DOSSIER}/train.csv", dtype={"id": str})
val = pd.read_csv(f"{DOSSIER}/val.csv", dtype={"id": str})

print("Train :", train.shape, "| Validation :", val.shape)
print("transformers :", transformers.__version__)
print("GPU :", torch.cuda.device_count(), "x",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")

Train : (12132, 11) | Validation : (2609, 11)
transformers : 5.20.2
GPU : 2 x Tesla T4


In [2]:
import re
from datasets import Dataset, Sequence, Value
from transformers import AutoTokenizer

NOM_MODELE = "allenai/scibert_scivocab_uncased"
MAX_LENGTH = 384

tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

def nettoyer_titre(titre):
    titre = re.sub(r"\$.*?\$", " formula ", titre)    # formules LaTeX
    return re.sub(r"\s+", " ", titre).strip()           # sauts de ligne en trop

def tokeniser(lot):
    # [CLS] titre [SEP] résumé [SEP] ; si trop long, on coupe le résumé, jamais le titre
    return tokenizer(lot["title"], lot["text"], truncation="only_second", max_length=MAX_LENGTH)

def vers_dataset(df, avec_labels=True):
    donnees = {"title": [nettoyer_titre(t) for t in df["title"]], "text": df["text"].tolist()}
    if avec_labels:
        donnees["labels"] = df[GROUPES].values.astype("float32").tolist()
    ds = Dataset.from_dict(donnees)
    if avec_labels:
        ds = ds.cast_column("labels", Sequence(Value("float32")))
    return ds.map(tokeniser, batched=True, remove_columns=["title", "text"])

ds_train = vers_dataset(train)
ds_val = vers_dataset(val)
print(ds_train)
print(tokenizer.decode(ds_train[0]["input_ids"][:40]))

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

Casting the dataset:   0%|          | 0/12132 [00:00<?, ? examples/s]

Map:   0%|          | 0/12132 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2609 [00:00<?, ? examples/s]

Map:   0%|          | 0/2609 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12132
})
[CLS] anisotropic thermo - elasticity in 2d - - part i : a unified approach [SEP] in this note we develop tools and techniques for the treatment of anisotropic thermo - elasticity in two space dimensions. we use


In [3]:
from transformers import AutoModelForSequenceClassification

modele = AutoModelForSequenceClassification.from_pretrained(
    NOM_MODELE, num_labels=len(GROUPES), problem_type="multi_label_classification",
    id2label=dict(enumerate(GROUPES)), label2id={g: i for i, g in enumerate(GROUPES)})

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  442MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

In [4]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def calculer_metriques(eval_pred):
    logits, labels = eval_pred
    probas = 1 / (1 + np.exp(-logits))
    return evaluer(labels.astype(int), probas_vers_predictions(probas))

NB_EPOCHS = 4
TAILLE_LOT = 16
nb_gpus = max(1, torch.cuda.device_count())
pas_par_epoch = math.ceil(len(ds_train) / (TAILLE_LOT * nb_gpus))
pas_warmup = int(0.1 * pas_par_epoch * NB_EPOCHS)
print(f"Pas par epoch : {pas_par_epoch} | pas de warmup : {pas_warmup}")

args = TrainingArguments(
    output_dir="/kaggle/working/scibert", num_train_epochs=NB_EPOCHS,
    learning_rate=2e-5, per_device_train_batch_size=TAILLE_LOT, per_device_eval_batch_size=64,
    warmup_steps=pas_warmup, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="micro_f1", save_total_limit=1,
    logging_strategy="epoch", report_to="none", seed=42)

trainer = Trainer(model=modele, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=calculer_metriques)
trainer.train()

Pas par epoch : 380 | pas de warmup : 152


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Hamming Loss,Subset Accuracy,Runtime,Samples Per Second,Steps Per Second
1,0.645533,0.415293,0.799411,0.795006,0.078335,0.557302,28.335900,92.074000,0.741000
2,0.358218,0.373219,0.817217,0.815349,0.072633,0.588348,28.311300,92.154000,0.742000
3,0.293158,0.364286,0.825810,0.824528,0.070621,0.597164,28.282100,92.249000,0.743000
4,0.249848,0.367683,0.824073,0.822074,0.071148,0.593714,28.303200,92.180000,0.742000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1520, training_loss=0.3866894571404708, metrics={'train_runtime': 1850.3396, 'train_samples_per_second': 26.227, 'train_steps_per_second': 0.821, 'total_flos': 9111566230788096.0, 'train_loss': 0.3866894571404708, 'epoch': 4.0})

In [5]:
pred_val = trainer.predict(ds_val)
probas_val = 1 / (1 + np.exp(-pred_val.predictions))
y_val = pred_val.label_ids.astype(int)

scores_05 = evaluer(y_val, probas_vers_predictions(probas_val))
seuils = chercher_seuils(y_val, probas_val)
scores_seuils = evaluer(y_val, probas_vers_predictions(probas_val, seuils))

print("Seuils choisis :", dict(zip(GROUPES, seuils.tolist())))
print(f"{'':16s} {'seuil 0.5':>10s} {'seuils/groupe':>14s}")
for nom in scores_05:
    print(f"{nom:16s} {scores_05[nom]:>10.4f} {scores_seuils[nom]:>14.4f}")
print()
print(rapport_par_groupe(y_val, probas_vers_predictions(probas_val, seuils)))

with open("/kaggle/working/scibert_titre_val.json", "w") as f:
    json.dump({"seuil_05": scores_05, "seuils": seuils.tolist(), "seuils_par_groupe": scores_seuils,
               "historique": trainer.state.log_history}, f, indent=2)
np.save("/kaggle/working/scibert_titre_val_probas.npy", probas_val)
print("Sauvegardé : scibert_titre_val.json, scibert_titre_val_probas.npy")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Seuils choisis : {'cs': 0.35, 'math': 0.35, 'physics': 0.3, 'q-bio': 0.6, 'q-fin': 0.4, 'stat': 0.55, 'eess': 0.25, 'econ': 0.6}
                  seuil 0.5  seuils/groupe
micro_f1             0.8258         0.8312
macro_f1             0.8245         0.8310
hamming_loss         0.0706         0.0711
subset_accuracy      0.5972         0.5899

              precision    recall  f1-score   support

          cs       0.81      0.90      0.85      1023
        math       0.75      0.80      0.77       606
     physics       0.84      0.86      0.85       623
       q-bio       0.90      0.88      0.89       389
       q-fin       0.88      0.89      0.88       436
        stat       0.79      0.71      0.75       484
        eess       0.78      0.86      0.82       383
        econ       0.81      0.86      0.83       366

   micro avg       0.81      0.85      0.83      4310
   macro avg       0.82      0.84      0.83      4310
weighted avg       0.82      0.85      0.83      4310
 samp

In [6]:
import shutil

# Le score test sera calculé UNE SEULE FOIS, à l'étape 8
test = pd.read_csv(f"{DOSSIER}/test.csv", dtype={"id": str})
ds_test = vers_dataset(test, avec_labels=False)
pred_test = trainer.predict(ds_test)
probas_test = 1 / (1 + np.exp(-pred_test.predictions))
np.save("/kaggle/working/scibert_titre_test_probas.npy", probas_test)
test[["id"]].to_csv("/kaggle/working/scibert_titre_test_ids.csv", index=False)
print("Probabilités test sauvegardées :", probas_test.shape)

shutil.rmtree("/kaggle/working/scibert", ignore_errors=True)   # supprime les checkpoints

Map:   0%|          | 0/2608 [00:00<?, ? examples/s]

Probabilités test sauvegardées : (2608, 8)
